# Exercise 06 Convert CloudCompare exports for QGIS

Use this notebook for step 2.4. It changes column names so QGIS can use
your CloudCompare results. It preserves every data row and numerical
value, including missing values; it does not calculate distances,
correct alignment, or apply a detection threshold.

No local Python installation or separate script upload is needed.
Sign in to Google Colab, click **Connect**, then run the three code cells
below in order using the triangle beside each cell. Use the default
CPU runtime; no GPU is needed. Your selected files are uploaded to the
Colab runtime. This notebook does not request access to Google Drive.

Before starting, export **m3c2.asc** and **c2c.asc** from CloudCompare:
choose ASCII cloud, comma separator, column-name header, and no
point-count header. Keep coordinates and scalar fields. Export the
M3C2 output cloud for the first file and the June cloud carrying
C2C absolute distances for the second.

## 1 Upload your two exports

Run the cell and click **Choose Files**. Select both **m3c2.asc** and
**c2c.asc**, then wait for both uploads to finish. Do not upload the LAS
files, the companion ZIP, or a Python script. If either filename is
different, rename your export before uploading.

In [ ]:
from google.colab import files

uploaded = files.upload()
required_files = ("m3c2.asc", "c2c.asc")
missing = [name for name in required_files if name not in uploaded]
if missing:
    raise ValueError(
        "Missing " + ", ".join(missing)
        + ". Run this cell again and choose both exports."
    )
print("Both exports uploaded. Run step 2 next.")

## 2 Convert the column names

Run the cell below. A successful conversion prints the number of rows
preserved in each CSV. An error about missing fields means you should
check which cloud you exported and whether you included scalar fields
and a column-name header. Do not continue to step 3 after an error.

Each run creates a separate output folder, preserving earlier CSVs.

In [ ]:
import csv
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

archive_path = None
if "uploaded" not in globals():
    raise ValueError("Run step 1 and upload both exports first.")
output_folder = Path(tempfile.mkdtemp(prefix="exercise06_conversion_"))
helper_path = output_folder / "normalize_export.py"

helper_path.write_text('"""Convert a CloudCompare comma-delimited ASCII export to a QGIS CSV.\n\nUsage: python3 normalize_export.py input.asc output.csv\nPreserves every row and every numerical string, including nan. Renames the\nCloudCompare scalar fields; no filtering, bias correction, or interpolation.\n"""\nfrom pathlib import Path\nimport argparse\nimport csv\n\nFIELD_NAMES = {\n    "M3C2 distance": "M3C2_distance", "distance uncertainty": "LoD95",\n    "significant change": "significant", "STD_cloud1": "STD_jan09",\n    "STD_cloud2": "STD_jun10", "Npoints_cloud1": "N_jan09",\n    "Npoints_cloud2": "N_jun10", "C2C absolute distances": "C2C_distance",\n}\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("input", type=Path)\n    parser.add_argument("output", type=Path)\n    args = parser.parse_args()\n    if args.input.resolve() == args.output.resolve() or args.output.exists():\n        parser.error("Choose a new output file; existing files are preserved.")\n    count = 0\n    with args.input.open(newline="") as source, args.output.open("x", newline="") as target:\n        reader, writer = csv.reader(source), csv.writer(target)\n        fields = next(reader)\n        fields[0] = fields[0].removeprefix("//")\n        if fields[:3] != ["X", "Y", "Z"]:\n            raise ValueError("Expected header X,Y,Z and a comma-separated export with column names.")\n        writer.writerow([FIELD_NAMES.get(f, f) for f in fields])\n        for row in reader:\n            if len(row) != len(fields):\n                raise ValueError(f"Row {count+2}: field count differs from header")\n            writer.writerow(row)\n            count += 1\n    print(f"Preserved {count} rows in {args.output}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')


expected_fields = {
    "m3c2.asc": ("M3C2 distance", "distance uncertainty", "significant change"),
    "c2c.asc": ("C2C absolute distances",),
}
converted_files = []
for filename, expected in expected_fields.items():
    if filename not in uploaded or not Path(filename).is_file():
        raise ValueError("Run step 1 and upload both exports first.")
    with Path(filename).open(newline="") as stream:
        header = next(csv.reader(stream), [])
    absent = [field for field in expected if field not in header]
    if absent:
        raise ValueError(
            filename + " is missing " + ", ".join(absent)
            + ". Check the selected cloud and ASCII export options."
        )
    output = output_folder / Path(filename).with_suffix(".csv").name
    result = subprocess.run(
        [sys.executable, str(helper_path), filename, str(output)],
        capture_output=True, text=True,
    )
    if result.returncode:
        raise RuntimeError(result.stderr.strip())
    message = result.stdout.strip().split(" in ")[0]
    print(output.name + ": " + message)
    converted_files.append(output)

archive = output_folder / "exercise06_qgis_csvs.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for output in converted_files:
        bundle.write(output, arcname=output.name)
archive_path = archive
print("Conversion complete. Run step 3 to download both CSV files.")

## 3 Download your results

Run the cell below to download **exercise06_qgis_csvs.zip**. Extract
the ZIP on your computer: it contains **m3c2.csv** and **c2c.csv**.
Download the files before leaving Colab; runtime files are temporary.

Continue with Part 4 in QGIS. Import each CSV as Delimited Text with
**X field = X**, **Y field = Y**, and **EPSG:26913**. Check that the
coordinates are around **X = 386000 m** and **Y = 3632800 m**. If they
are small shifted coordinates, correct the CloudCompare export first;
this notebook does not change coordinates.

The M3C2 CSV uses **M3C2_distance**, **LoD95**, and **significant**.
LoD95 is the exported distance uncertainty, already a detection
threshold. The C2C CSV uses **C2C_distance**.

In [ ]:
if "archive_path" not in globals() or archive_path is None or not archive_path.is_file():
    raise RuntimeError("Complete step 2 successfully before downloading.")
files.download(str(archive_path))